In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from tqdm import tqdm 
import re

In [2]:
file_path1 = "/kaggle/input/twitter/PNAS_DE_politician_tweets_2016-01-01_to_2022-03-16.csv/PNAS_DE_politician_tweets_2016-01-01_to_2022-03-16.csv.gzip"


df1 = pd.read_csv(file_path1)


In [3]:
df1

,id,author_id,created_at,retweeted,quoted,reply,has_url,retweet_count,reply_count,like_count,...,C_3,C_4,C_5,C_6,C_7,C_8,fishy_20,fishy_40,fishy_60,unreliable
0,1503680985224949766,1419751204465356805,2022-03-15 10:34:18+00:00,False,False,False,True,3,9,41,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1503325058781044739,1419751204465356805,2022-03-14 10:59:59+00:00,False,False,True,False,4,6,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1503324790811205635,1419751204465356805,2022-03-14 10:58:55+00:00,False,False,False,False,8,5,46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1503071011461283841,1419751204465356805,2022-03-13 18:10:29+00:00,False,False,True,False,2,2,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1503030431985328137,1419751204465356805,2022-03-13 15:29:14+00:00,False,True,False,True,18,7,93,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972265,692693500907880448,16674128,2016-01-28 12:59:34+00:00,False,False,False,True,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
972266,692372805539741697,16674128,2016-01-27 15:45:15+00:00,False,False,False,True,45,4,77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
972267,692034440298676224,16674128,2016-01-26 17:20:42+00:00,False,False,False,True,4,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
972268,688976576088768512,16674128,2016-01-18 06:49:50+00:00,False,False,False,True,11,4,20,...,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,NaN


In [4]:
df1.keys()

Index(['id', 'author_id', 'created_at', 'retweeted', 'quoted', 'reply',
       'has_url', 'retweet_count', 'reply_count', 'like_count', 'quote_count',
       'party', 'Score', 'transparency', 'accuracy', 'C_0', 'C_1', 'C_2',
       'C_3', 'C_4', 'C_5', 'C_6', 'C_7', 'C_8', 'fishy_20', 'fishy_40',
       'fishy_60', 'unreliable'],
      dtype='object')

In [5]:
file_path2 = "/kaggle/input/twitter/PNAS_combined_DE_politician_twitter_timelines_2016-01-01_to_2022-03-16_clean.csv/PNAS_combined_DE_politician_twitter_timelines_2016-01-01_to_2022-03-16_clean.csv.gzip"

df2 = pd.read_csv(file_path2)

In [6]:
df2.keys()

Index(['id', 'author_id', 'created_at', 'expanded_urls', 'retweeted', 'quoted',
       'reply', 'text', 'retweet_count', 'reply_count', 'like_count',
       'quote_count'],
      dtype='object')

In [7]:
file_path3 = "/kaggle/input/twitter/combined_DE_politician_twitter_timelines_2016-01-01_to_2023-02-11_clean.csv/combined_DE_politician_twitter_timelines_2016-01-01_to_2023-02-11_clean.csv.gzip"

df3 = pd.read_csv(file_path3)

In [8]:
df3

,id,author_id,created_at,expanded_urls,retweeted,quoted,reply,text,retweet_count,reply_count,like_count,quote_count
0,1623371262864068617,1419751204465356805,2023-02-08 17:20:44+00:00,[],False,False,False,"Shell, ExxonMobil, Total Energies, sie alle ha...",10,8,70,0
1,1622589284002717700,1419751204465356805,2023-02-06 13:33:26+00:00,['https://twitter.com/FBsirske/status/16225892...,False,False,False,"Die Post streikt!\n\nDie Post macht 8,4 Millia...",12,5,111,0
2,1621139288103735297,1419751204465356805,2023-02-02 13:31:40+00:00,['https://twitter.com/kimvsparrentak/status/16...,False,True,False,Uber und Co. hintergehen geltendes Arbeitsrech...,5,2,15,0
3,1620521818292617217,1419751204465356805,2023-01-31 20:38:03+00:00,[],False,False,False,"Die Überlegungen aus der FDP, wegen des #Soli ...",4,9,52,0
4,1620357371506884610,1419751204465356805,2023-01-31 09:44:36+00:00,['https://www.tagesschau.de/eilmeldung/solidar...,False,False,False,Spitzenverdiener zur Kasse! \n\nDer #Soli nütz...,3,0,14,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1559354,2004061800894464,16674128,2010-11-09 14:26:20+00:00,[],False,False,False,"""get me on the court and I'm trouble..."" (Ice ...",0,0,0,0
1559355,1934566277251072,16674128,2010-11-09 09:50:11+00:00,[],False,False,False,"Ohrwurm: ""Hey mister tambourine-man..."" von Di...",0,0,0,0
1559356,1004139857317889,16674128,2010-11-06 20:13:00+00:00,[],False,False,False,grinst bei jedem Anblick von VW-Fahrzeugen all...,1,0,0,0
1559357,847738329632768,16674128,2010-11-06 09:51:31+00:00,[],False,False,False,Grüße von der Listenaufstellung in Frankfurt z...,0,0,0,0


In [9]:
# Selecting only the relevant columns from df1 and df2 before merging
df1_selected = df1[['id', 'author_id', 'created_at', 'party']]
df2_selected = df2[['id', 'reply', 'text', 'retweet_count', 'reply_count', 'like_count', 'expanded_urls', 'retweeted', 'quote_count']]

# Merging the selected columns from both DataFrames
merged_df = pd.merge(df1_selected, df2_selected, on='id', how='inner')
# Dropping rows where 'party' is NaN or 'Independent'
merged_df = merged_df[~merged_df['party'].isin(['Independent', np.nan])]
merged_df

,id,author_id,created_at,party,reply,text,retweet_count,reply_count,like_count,expanded_urls,retweeted,quote_count
0,1503680985224949766,1419751204465356805,2022-03-15 10:34:18+00:00,Alliance 90/The Greens,False,Die Planungen zum #Infektionsschutzgesetz sehe...,3,9,41,['https://twitter.com/FBsirske/status/15036809...,False,0
1,1503325058781044739,1419751204465356805,2022-03-14 10:59:59+00:00,Alliance 90/The Greens,True,Was stattdessen wirklich hilft: Ein #Energiege...,4,6,34,[],False,0
2,1503324790811205635,1419751204465356805,2022-03-14 10:58:55+00:00,Alliance 90/The Greens,False,Menschen kämpfen mit ihren Rechnungen für Heiz...,8,5,46,[],False,0
3,1503071011461283841,1419751204465356805,2022-03-13 18:10:29+00:00,Alliance 90/The Greens,True,@IngeHannemann Ich setze mich für ein Energieg...,2,2,8,[],False,0
4,1503030431985328137,1419751204465356805,2022-03-13 15:29:14+00:00,Alliance 90/The Greens,False,Wir brauchen schnell eine bessere soziale Entl...,18,7,93,['https://twitter.com/wazwolfsburg/status/1502...,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...
972265,692693500907880448,16674128,2016-01-28 12:59:34+00:00,Alliance 90/The Greens,False,"Die Schmerzen eines Köln-Fans, angesprochenes ...",0,0,0,['https://koalitionrutwiess.files.wordpress.co...,False,0
972266,692372805539741697,16674128,2016-01-27 15:45:15+00:00,Alliance 90/The Greens,False,Bewegende Rede v Prof. Dr. Ruth Klüger im #Bun...,45,4,77,"['http://youtu.be/-K02wZPcrLM', 'https://twitt...",False,1
972267,692034440298676224,16674128,2016-01-26 17:20:42+00:00,Alliance 90/The Greens,False,@MSF #kefayawar #Yemen #300DaysOfWar and still...,4,0,2,['https://twitter.com/nouripour/status/6920344...,False,0
972268,688976576088768512,16674128,2016-01-18 06:49:50+00:00,Alliance 90/The Greens,False,Menschenrechtslage im #Iran nicht wegen Atomde...,11,4,20,['http://www.deutschlandfunk.de/atomabkommen-d...,False,0


In [10]:
# Extracting all unique values in the 'party' column
unique_parties = merged_df['party'].unique()

unique_parties

array(['Alliance 90/The Greens', 'CDU/CSU', 'SPD', 'The Left Party',
       'AFD', 'CDU', 'FDP', 'CSU'], dtype=object)

In [11]:
# Mapping parties to political labels: 'Right' for AfD, 'Left' for The Left Party, and 'Middle' for the rest
party_mapping = {
    'AFD': 'Right',
    'The Left Party': 'Left',
    'Alliance 90/The Greens': 'Middle',
    'CDU/CSU': 'Middle',
    'SPD': 'Middle',
    'CDU': 'Middle',
    'FDP': 'Middle',
    'CSU': 'Middle'
}

# Applying the mapping to a 'political_label' column
merged_df['leaning'] = merged_df['party'].map(party_mapping)

merged_df.to_csv("twitter.csv", index = False)

In [12]:
# Preprocessing Function
def preprocess(df):

    # Load stopwords
    stopwords_path = "/kaggle/input/final-data/stopwords_youtube.txt"
    try:
        with open(stopwords_path, "r", encoding="utf-8") as file:
            loaded_stopwords_set = set(line.strip() for line in file)
    except FileNotFoundError:
        print("stopwords not found")
        loaded_stopwords_set = set()

    
    tqdm.pandas()
    
    def remove_special_characters(text):
        pattern = r'[^a-zA-ZäöüÄÖÜß\s]'
        text = re.sub(pattern, '', text)
        text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
        return text
    
    def remove_stopwords(text, stopwords_set):
        text = str(text)
        return ' '.join([word for word in text.split() if word.lower() not in stopwords_set and len(word) >= 3])
    
    # Drop rows where 'transcript' is NaN
    df = df.dropna(subset=['text'])
    
    # Convert the 'transcript' column to strings
    df['text_clean'] = df['text'].astype(str)
    
    df['text_clean'] = df['text_clean'].progress_apply(remove_special_characters)
    df['text_clean'] = df['text_clean'].progress_apply(lambda x: remove_stopwords(x, loaded_stopwords_set))
    df['length'] = df['text_clean'].progress_apply(lambda x: len(x.split()))

    print(f"Average Speech Length: {df['length'].mean():.2f}")
    return df

In [13]:
merged_df = preprocess(merged_df)

100%|██████████| 922146/922146 [00:01<00:00, 475116.58it/s]

Average Speech Length: 10.21


In [14]:
merged_df

,id,author_id,created_at,party,reply,text,retweet_count,reply_count,like_count,expanded_urls,retweeted,quote_count,leaning,text_clean,length
0,1503680985224949766,1419751204465356805,2022-03-15 10:34:18+00:00,Alliance 90/The Greens,False,Die Planungen zum #Infektionsschutzgesetz sehe...,3,9,41,['https://twitter.com/FBsirske/status/15036809...,False,0,Middle,Planungen Infektionsschutzgesetz sehen CoronaM...,17
1,1503325058781044739,1419751204465356805,2022-03-14 10:59:59+00:00,Alliance 90/The Greens,True,Was stattdessen wirklich hilft: Ein #Energiege...,4,6,34,[],False,0,Middle,stattdessen hilft Energiegeld Menschen ausgeza...,8
2,1503324790811205635,1419751204465356805,2022-03-14 10:58:55+00:00,Alliance 90/The Greens,False,Menschen kämpfen mit ihren Rechnungen für Heiz...,8,5,46,[],False,0,Middle,Menschen Rechnungen Heizung Sprit Strom Tankra...,15
3,1503071011461283841,1419751204465356805,2022-03-13 18:10:29+00:00,Alliance 90/The Greens,True,@IngeHannemann Ich setze mich für ein Energieg...,2,2,8,[],False,0,Middle,IngeHannemann setze Energiegeld pro Kopf Bürge...,11
4,1503030431985328137,1419751204465356805,2022-03-13 15:29:14+00:00,Alliance 90/The Greens,False,Wir brauchen schnell eine bessere soziale Entl...,18,7,93,['https://twitter.com/wazwolfsburg/status/1502...,False,0,Middle,brauchen schnell bessere soziale Entlastung En...,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972265,692693500907880448,16674128,2016-01-28 12:59:34+00:00,Alliance 90/The Greens,False,"Die Schmerzen eines Köln-Fans, angesprochenes ...",0,0,0,['https://koalitionrutwiess.files.wordpress.co...,False,0,Middle,Schmerzen KölnFans angesprochenes Grußwort nht...,9
972266,692372805539741697,16674128,2016-01-27 15:45:15+00:00,Alliance 90/The Greens,False,Bewegende Rede v Prof. Dr. Ruth Klüger im #Bun...,45,4,77,"['http://youtu.be/-K02wZPcrLM', 'https://twitt...",False,1,Middle,Bewegende Rede Prof Ruth Klüger Bundestag Jahr...,11
972267,692034440298676224,16674128,2016-01-26 17:20:42+00:00,Alliance 90/The Greens,False,@MSF #kefayawar #Yemen #300DaysOfWar and still...,4,0,2,['https://twitter.com/nouripour/status/6920344...,False,0,Middle,MSF kefayawar Yemen DaysOfWar still bombing ht...,7
972268,688976576088768512,16674128,2016-01-18 06:49:50+00:00,Alliance 90/The Greens,False,Menschenrechtslage im #Iran nicht wegen Atomde...,11,4,20,['http://www.deutschlandfunk.de/atomabkommen-d...,False,0,Middle,Menschenrechtslage Iran Atomdeal ignorieren ht...,7


In [15]:
merged_df.to_csv("twitter.csv", index = False)